In [35]:
import torch
import os

In [36]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

# Preparing our Dataset

In [37]:

from torchvision.io import decode_image ,ImageReadMode


labels_map = {
    0: "NORMAL",
    1: "PNEUMONIA",
}   
class_to_idx = {
    "NORMAL":0,
    "PNEUMONIA":1
}

class XrayDataset(Dataset):
    def __init__(self, train_dir , transform = None , target_transform=None):
        self.img_samples = self.annotation_pair(train_dir)
        self.transform = transform
        self.target_transform = target_transform

    def annotation_pair(self,train_dir):
        samples = []
        for className in os.listdir(train_dir):
            for filename in os.listdir(os.path.join(train_dir,className)):
                samples.append((os.path.join(train_dir,className,filename),class_to_idx[className]))
        return samples  

    def __len__(self):
        return len(self.img_samples)

    def __getitem__(self,idx):
        img_path = self.img_samples[idx][0]
        label = self.img_samples[idx][1]
        image = decode_image(img_path , mode=ImageReadMode.GRAY)
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        
        return image , label
    

# Loading our Dataset

In [38]:
from torch.utils.data import DataLoader
from torchvision.transforms import v2


train_dir = "/kaggle/input/datasets/mahabubalam31/chest-x-ray-dataset-for-pneumoniabalanced/dataset/TRAIN"
test_dir = "/kaggle/input/datasets/mahabubalam31/chest-x-ray-dataset-for-pneumoniabalanced/dataset/TEST"
training_data = XrayDataset(train_dir,transform=v2.Compose(
    [v2.ToImage(),
     v2.Resize((224, 224)) ,
     v2.RandomHorizontalFlip(p=0.5),
     v2.RandomRotation(degrees=10),
     v2.ColorJitter(brightness=0.2, contrast=0.2),
     v2.RandomAffine(degrees=0, translate=(0.05, 0.05)),
     v2.ToDtype(torch.float32, scale=True)]
))
test_data = XrayDataset(test_dir,transform=v2.Compose(
    [v2.ToImage(),
     v2.Resize((224, 224)),
     v2.ToDtype(torch.float32, scale=True)]
))


train_dataloader = DataLoader(training_data, batch_size=39, shuffle=True , num_workers=4, pin_memory=True)
test_dataloader = DataLoader(test_data, batch_size=39, shuffle=True, num_workers=4, pin_memory=True)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"



In [39]:
print(train_dataloader)
X, y = next(iter(train_dataloader))
print(X.shape)
print(y.shape)
print(f"We have {(len(train_dataloader))} batches in total" )

torch.Size([39, 1, 224, 224])
torch.Size([39])
We have 234 batches in total


# Building Our Neural Net PneumoNet V1.0

In [40]:
import torch.nn as nn
# class PneumoNet(nn.Module): version_1
#     def __init__(self):
#         super().__init__()
#         self.flatten = nn.Flatten()
#         self.stack  = nn.Sequential(
#             nn.Linear(224*224,256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 32),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(32, 1),
#         )
#     def forward(self,x):
#         x = self.flatten(x)
#         logits = self.stack(x)
#         return logits
class PneumoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),    
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),    
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),         
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 28 -> 14, now 128 channels

            nn.AdaptiveAvgPool2d((1,1)),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128,32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32,1)
        )
    def forward(self,x):
        x = self.conv_stack(x)
        logits = self.fc(x)
        
        return logits
model = PneumoNet().to(device)

In [41]:
print(model)

total_params = sum(p.numel() for p in model.parameters())

PneumoNet(
  (conv_stack): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [42]:
print(total_params)

101793


# Defining Hyperparameters

In [43]:
lr = 1e-4
batch_size = 39
epochs = 20

# Defining Optimizer and loss funciton

In [44]:
from torch import optim

# pos_weight = torch.tensor([1349 / 3883]).to(device)
# loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
loss_fn = nn.BCEWithLogitsLoss()
optimizer =  optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# Building Train Loop

In [45]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

def training_loop(dataloader , model,  optimizer ,  loss_fn) : 
    size = len(dataloader.dataset)
    model.train()

    for batch , (X, y ) in enumerate(dataloader):
        X, y = X.to(device) , y.to(device)
        pred = model(X)
        loss = loss_fn(pred.squeeze(),y.float())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        
        if batch % 20 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f" Current LR: {current_lr}")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred.squeeze(), y.float()).item()
            correct += ((pred.squeeze() > 0).float() == y).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss

In [46]:
best_loss = float('inf')
for i in range(epochs):
    print(f"Epoch {i+1}\n-------------------------------")
    training_loop(train_dataloader, model, optimizer, loss_fn)
    test_loss = test_loop(test_dataloader,model,loss_fn)
    if test_loss < best_loss:
        best_loss = test_loss
        torch.save(model.state_dict(), "best_fc_model.pt")
print("Training Complete")

Epoch 1
-------------------------------
loss: 0.717559  [   39/ 9126]
loss: 0.661388  [  819/ 9126]
loss: 0.689713  [ 1599/ 9126]
loss: 0.634282  [ 2379/ 9126]
loss: 0.625892  [ 3159/ 9126]
loss: 0.597205  [ 3939/ 9126]
loss: 0.667421  [ 4719/ 9126]
loss: 0.653258  [ 5499/ 9126]
loss: 0.632820  [ 6279/ 9126]
loss: 0.637788  [ 7059/ 9126]
loss: 0.586770  [ 7839/ 9126]
loss: 0.581436  [ 8619/ 9126]
 Current LR: 0.0001
Test Error: 
 Accuracy: 73.9%, Avg loss: 0.569628 

Epoch 2
-------------------------------
loss: 0.622391  [   39/ 9126]
loss: 0.603381  [  819/ 9126]
loss: 0.576800  [ 1599/ 9126]
loss: 0.545258  [ 2379/ 9126]
loss: 0.648784  [ 3159/ 9126]
loss: 0.629839  [ 3939/ 9126]
loss: 0.472256  [ 4719/ 9126]
loss: 0.416789  [ 5499/ 9126]
loss: 0.613714  [ 6279/ 9126]
loss: 0.533078  [ 7059/ 9126]
loss: 0.645677  [ 7839/ 9126]
loss: 0.438811  [ 8619/ 9126]
 Current LR: 0.0001
Test Error: 
 Accuracy: 73.3%, Avg loss: 0.551741 

Epoch 3
-------------------------------
loss: 0.440683  

In [47]:
import torch
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# model = PneumoNet().to(device)
# model.load_state_dict(torch.load("best_fc_model.pt"))
# model.eval()

# predictor =  model

all_preds, all_labels = [], []

with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        pred_labels = (pred.squeeze() > 0).float()   # sign of logit -> 0/1 prediction
        all_preds.extend(pred_labels.cpu().tolist())
        all_labels.extend(y.cpu().tolist())

In [48]:
cm = confusion_matrix(all_labels, all_preds)
print(cm)

print(classification_report(all_labels, all_preds, target_names=["NORMAL", "PNEUMONIA"]))

[[520  51]
 [131 439]]
              precision    recall  f1-score   support

      NORMAL       0.80      0.91      0.85       571
   PNEUMONIA       0.90      0.77      0.83       570

    accuracy                           0.84      1141
   macro avg       0.85      0.84      0.84      1141
weighted avg       0.85      0.84      0.84      1141



In [49]:
torch.save(model.state_dict(), "PneumoNetWithCNNv2.pt")